# Compute metrics (area, intensity)

Per-frame area and intensity readouts for each ROI in a timelapse experiment.

Inputs are OME-TIFF stacks with the channel layout produced by step `06_Stack_seg_apply_masks`:

| Index | Channel |
| :---: | :--- |
| 0 | CAAX (raw) |
| 1 | cell membrane (raw) |
| 2 | actin (background-subtracted, ratio-corrected) |
| 3 | compacted-region segmentation (binary) |
| 4 | CAAX-positive segmentation (binary) |

Output is `analysis.csv` in the experiment's `tables/` directory, with one row per (ROI, frame). Columns include per-region areas (in microns squared), per-channel mean and median intensity under cell / compacted / CAAX-positive masks, total cell area, percent compaction, and integrated densities.

In [ ]:

import bioio_ome_tiff


In [ ]:
# Directory containing the per-ROI OME-TIFF stacks output by 06_Stack_seg_apply_masks.
input_dirpath = Path(input('Path to ROI image directory: '))

In [ ]:
# Channel layout in the input OME-TIFF stack.
CAAX_CH = 0
CELL_CH = 1
ACTIN_CH = 2
SEG_CMP_CH = 3
SEG_CAAX_CH = 4

In [ ]:
ROI_names = sorted(path.name for path in input_dirpath.glob('*.ome.tif'))

proc_dirpath = utils.get_proc_dirpath(input_dirpath)
analysis_df_path = proc_dirpath / dn.tables_dirname / 'analysis.csv'
analysis_df_path.parent.mkdir(parents=True, exist_ok=True)

analysis_df = pd.DataFrame({'ROI imgname': ROI_names})
utils.safe_save_csv(analysis_df, analysis_df_path)

In [ ]:
size_t = None
for ROI_name in tqdm(ROI_names):
    imgpath = input_dirpath / ROI_name
    img_file = BioImage(imgpath, reader=bioio_ome_tiff.Reader)
    img = img_file.data

    # On the first image, expand the dataframe to one row per (ROI, frame).
    if size_t is None:
        size_t = img.shape[0]
        analysis_df['size_t'] = size_t
        analysis_df = (
            analysis_df.loc[analysis_df.index.repeat(analysis_df['size_t'])]
            .reset_index(drop=True)
        )
        analysis_df['t'] = analysis_df.groupby('ROI imgname').cumcount()

    df_idc = analysis_df.index[analysis_df['ROI imgname'] == imgpath.name].tolist()

    # Areas of the two segmentation channels in microns squared.
    seg = img[:, [SEG_CMP_CH, SEG_CAAX_CH], :, :, :]
    seg_labels = ['compacted', 'CAAX-positive']
    pixel_area = img_file.physical_pixel_sizes.X * img_file.physical_pixel_sizes.Y
    analysis_df = compute_areas(seg, seg_labels, df_idc, analysis_df, pixel_area)

    # Mean and median intensity of the CAAX and actin channels under three
    # masks: full cell, compacted region, CAAX-positive region.
    img_subset = img[:, [CAAX_CH, ACTIN_CH], :, :, :]
    ch_labels = ['caax', 'actin']
    seg_cmp = img[:, [SEG_CMP_CH], :, :, :]
    seg_caax = img[:, [SEG_CAAX_CH], :, :, :]
    seg_cell = seg_cmp | seg_caax

    analysis_df = compute_int(img_subset, ch_labels, df_idc, analysis_df, mask=seg_cell, mask_label='cell')
    analysis_df = compute_int(img_subset, ch_labels, df_idc, analysis_df, mask=seg_cmp, mask_label='compacted')
    analysis_df = compute_int(img_subset, ch_labels, df_idc, analysis_df, mask=seg_caax, mask_label='caax')

    analysis_df.to_csv(analysis_df_path, index=False)

print('Done!')

In [ ]:
# Per-cell summary metrics: total cell area, % compaction, integrated densities.
analysis_df = derive_compaction_metrics(analysis_df, integrated_density_channels=['caax', 'actin'])
analysis_df.to_csv(analysis_df_path, index=False)